# Cropland final regime model

このノートブックは、`raw_crop_wx`を基準に、複数の農業レジームに分けた場合に空間OOF精度が改善するかを検証するための最小構成です。

実行順：上から順番に実行してください。旧SHAP分析・Moran's I・不要な比較図は含めていません。

レジーム別モデルの採用判断は、最後のセルで表示される`delta_r2`とfold別の安定性で行います。


## 【セル1】データ読み込み・共通設定


In [ ]:
from __future__ import annotations

import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
from sklearn.neighbors import BallTree
from lightgbm import LGBMClassifier, LGBMRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(r"C:\masterresearch\Comparative_advantage")
GAEZ_DIR = ROOT / "GAEZ"
HYDE_DIR = ROOT / "HYDE3.4"
CROPLAND_DIR = HYDE_DIR / "cropland_npys"
DIST_DIR = ROOT / "distance_to_cities"
GLOFAS_DIR = ROOT / "GloFAS" / "processed_5min"
FEATURE_CACHE = GAEZ_DIR / "CroplandRegression" / "features_cache"
OUTPUT_DIR = GAEZ_DIR / "CroplandRegression" / "spatial_wx_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
YEAR = 2024
PRESENCE_THRESHOLD = 0.01
N_POS_SAMPLE = 120_000
N_ZERO_SAMPLE = 120_000
N_SPLITS = 5

WX_RADII_KM = (50, 100)
WX_SOURCE_FEATURES = [
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_rainfed_value_top5",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_2020",
]
INCLUDE_SLOPE_WX = False
MIN_VALID_NEIGHBOR_FRACTION = 0.25
SAVE_WX_RASTERS = True

EARTH_RADIUS_KM = 6371.0088
MORAN_K = 8
MORAN_MAX_N = 50_000


def load_cache(*names):
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            print("load cache:", path.name)
            return np.load(path, mmap_mode="r")
    raise FileNotFoundError(
        "Required cache not found: " + ", ".join(names)
    )


def safe_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    values = np.where(np.isfinite(values) & (values > 0), values, 0.0)
    return np.log1p(values).astype(np.float32)


lat = np.load(CROPLAND_DIR / "lat.npy")
lon = np.load(CROPLAND_DIR / "lon.npy")
SHAPE = (len(lat), len(lon))

cropland_raw = np.load(
    CROPLAND_DIR / "cropland_fraction_1950_2024.npy",
    mmap_mode="r",
)
years = np.load(CROPLAND_DIR / "years.npy")
year_index = int(np.where(years == YEAR)[0][0])
cropland_2024 = np.asarray(cropland_raw[year_index], dtype=np.float32).copy()
cropland_2024[~np.isfinite(cropland_2024)] = np.nan

population_density_2024 = load_cache("population_density_2024.npy")
elevation_m = load_cache("elevation_5min.npy")
slope = load_cache("slope_5min.npy")
exclusion = load_cache("exclusion_5min_mode.npy")
city_time_20k_min = np.load(DIST_DIR / "cities_10_1_12deg_min.npy", mmap_mode="r")
port_time_any_min = np.load(DIST_DIR / "ports_05_1_12deg_min.npy", mmap_mode="r")
glofas_p10 = np.load(GLOFAS_DIR / "p10_discharge_max_5min_2020.npy", mmap_mode="r")
distance_river_gt10 = np.load(
    GLOFAS_DIR / "distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy",
    mmap_mode="r",
)
rainfed_value_top5 = load_cache(
    "rainfed_value_top5_usd_per_ha_checked_36crops.npy",
    "rainfed_value_top5_checked_36crops.npy",
)
irrigated_value_top5 = load_cache(
    "irrigated_value_top5_usd_per_ha_checked_36crops.npy",
    "irrigated_value_top5_checked_36crops.npy",
)
rainfed_calorie_top5 = load_cache(
    "rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "rainfed_calorie_top5_checked_36crops.npy",
)
irrigated_calorie_top5 = load_cache(
    "irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "irrigated_calorie_top5_checked_36crops.npy",
)

land_mask = (
    np.isfinite(elevation_m)
    & np.isfinite(cropland_2024)
    & np.isfinite(population_density_2024)
    & (population_density_2024 >= 0)
)
presence = land_mask & (cropland_2024 > PRESENCE_THRESHOLD)

rng = np.random.default_rng(RANDOM_SEED)
pos_flat = np.flatnonzero(presence.ravel())
zero_flat = np.flatnonzero((land_mask & ~presence).ravel())
pos_sample = rng.choice(pos_flat, min(N_POS_SAMPLE, len(pos_flat)), replace=False)
zero_sample = rng.choice(zero_flat, min(N_ZERO_SAMPLE, len(zero_flat)), replace=False)
sample_flat = np.concatenate([pos_sample, zero_sample])
rng.shuffle(sample_flat)
rows, cols = np.unravel_index(sample_flat, SHAPE)


def take(array):
    return np.asarray(array[rows, cols])


sample = pd.DataFrame({
    "row": rows.astype(np.int32),
    "col": cols.astype(np.int32),
    "lat": lat[rows].astype(np.float32),
    "lon": lon[cols].astype(np.float32),
    "cropland_fraction": take(cropland_2024).astype(np.float32),
    "presence": (take(cropland_2024) > PRESENCE_THRESHOLD).astype(np.uint8),
    "elevation_m": take(elevation_m).astype(np.float32),
    "slope": take(slope).astype(np.float32),
    "exclusion_class": np.nan_to_num(take(exclusion), nan=-1).astype(np.int16),
    "log_pop_density_2024": safe_log1p(take(population_density_2024)),
    "log_city_time_20k_min": safe_log1p(take(city_time_20k_min)),
    "log_port_time_any_min": safe_log1p(take(port_time_any_min)),
    "log_glofas_p10_2020": safe_log1p(take(glofas_p10)),
    "log_distance_river_gt10_2020": safe_log1p(take(distance_river_gt10)),
    "log_rainfed_value_top5": safe_log1p(take(rainfed_value_top5)),
    "log_rainfed_calorie_top5": safe_log1p(take(rainfed_calorie_top5)),
    "log_irrigation_value_gain_top5": safe_log1p(
        np.maximum(take(irrigated_value_top5) - take(rainfed_value_top5), 0)
    ),
    "log_irrigation_calorie_gain_top5": safe_log1p(
        np.maximum(take(irrigated_calorie_top5) - take(rainfed_calorie_top5), 0)
    ),
}).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

sample["spatial_block"] = (
    np.floor((sample["lat"] + 90) / 10).astype(int) * 36
    + np.floor((sample["lon"] + 180) / 10).astype(int)
)

base_feature_cols = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_2020",
    "log_rainfed_value_top5",
    "log_rainfed_calorie_top5",
    "log_irrigation_value_gain_top5",
    "log_irrigation_calorie_gain_top5",
]

print("grid:", SHAPE)
print("sample rows:", len(sample))
print("sample presence share:", float(sample["presence"].mean()))

## 【セル2】50km・100kmの周辺変数（WX）作成


In [ ]:
# 既存のWXキャッシュを読み込むだけ
# キャッシュがない場合は、ここでは再計算せずエラーにする。

sample_rows = sample["row"].to_numpy(dtype=int)
sample_cols = sample["col"].to_numpy(dtype=int)
wx_feature_cols = []

for radius_km in WX_RADII_KM:
    for source_name in WX_SOURCE_FEATURES:
        wx_name = f"wx_{radius_km}km_{source_name}"
        cache_path = OUTPUT_DIR / f"{wx_name}.npy"

        if not cache_path.exists():
            raise FileNotFoundError(
                "WXキャッシュがありません。再計算は行わず停止します: "
                f"{cache_path}"
            )

        print("load WX cache:", cache_path.name)
        wx_raster = np.load(cache_path, mmap_mode="r")

        if wx_raster.shape != SHAPE:
            raise ValueError(
                f"WXキャッシュのshapeが不正です: {cache_path} "
                f"{wx_raster.shape} != {SHAPE}"
            )

        sample[wx_name] = np.asarray(
            wx_raster[sample_rows, sample_cols],
            dtype=np.float32,
        )
        wx_feature_cols.append(wx_name)

print("WX feature count:", len(wx_feature_cols))


## 【セル3】分析サンプル・空間fold作成


In [ ]:
sample = (
    sample.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=base_feature_cols + wx_feature_cols)
    .reset_index(drop=True)
)
analysis_sample = sample.copy()
y_all = analysis_sample["presence"].to_numpy(dtype=np.uint8)
groups = analysis_sample["spatial_block"].to_numpy()

if analysis_sample["presence"].nunique() < 2:
    raise ValueError("Both presence classes are required.")
target_prior = float(presence.sum() / land_mask.sum())

feature_cols_by_model = {
    "baseline": base_feature_cols,
    "wx": base_feature_cols + wx_feature_cols,
}

try:
    cell_area_km2 = np.load(CROPLAND_DIR / "grid_area_km2.npy", mmap_mode="r")
    sample_area = np.asarray(
        cell_area_km2[
            analysis_sample["row"].to_numpy(int),
            analysis_sample["col"].to_numpy(int),
        ],
        dtype=float,
    )
    n_pos_pop = int(presence.sum())
    n_zero_pop = int(land_mask.sum() - presence.sum())
    n_pos_samp = int((y_all == 1).sum())
    n_zero_samp = int((y_all == 0).sum())
    area_weight = np.where(
        y_all == 1,
        n_pos_pop / max(n_pos_samp, 1),
        n_zero_pop / max(n_zero_samp, 1),
    ) * sample_area
except FileNotFoundError:
    print("grid_area_km2.npy not found; area weighting skipped.")
    area_weight = None

splits = list(
    GroupKFold(n_splits=N_SPLITS).split(
        analysis_sample,
        y_all,
        groups=groups,
    )
)
print("analysis rows:", len(analysis_sample))
print("spatial blocks:", analysis_sample["spatial_block"].nunique())
print("target prior:", target_prior)

In [ ]:
# raw再推定セルで必要になる重み設定
weight_specs = {"unweighted_sample": None}
if area_weight is not None:
    weight_specs["area_weighted_reweighted"] = area_weight


## 【セル4】二段階LightGBMの関数定義

baseline・WXのOOF再推定は省略し、後続のrawモデルとレジームモデルが使う関数だけを定義します。


In [ ]:
def adjust_probability_prior_shift(probability, sample_prior, target_prior, eps=1e-6):
    probability = np.clip(np.asarray(probability, dtype=float), eps, 1 - eps)
    sample_prior = float(np.clip(sample_prior, eps, 1 - eps))
    target_prior = float(np.clip(target_prior, eps, 1 - eps))
    odds = probability / (1 - probability)
    odds *= (
        target_prior / (1 - target_prior)
    ) / (
        sample_prior / (1 - sample_prior)
    )
    return odds / (1 + odds)


def regression_summary(y_true, y_pred, weights=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]
    if weights is None:
        weights = np.ones(len(y_true), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)[valid]
    mean_true = np.average(y_true, weights=weights)
    residual = y_true - y_pred
    ss_res = np.sum(weights * residual**2)
    ss_tot = np.sum(weights * (y_true - mean_true)**2)
    return {
        "n": int(len(y_true)),
        "observed_mean": float(mean_true),
        "predicted_mean": float(np.average(y_pred, weights=weights)),
        "mae": float(np.average(np.abs(residual), weights=weights)),
        "rmse": float(np.sqrt(np.average(residual**2, weights=weights))),
        "r2": float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
        "bias_observed_minus_predicted": float(np.average(residual, weights=weights)),
    }


def classification_summary(y_true, probability, weights=None):
    y_true = np.asarray(y_true, dtype=np.uint8)
    probability = np.asarray(probability, dtype=float)
    valid = np.isfinite(probability)
    y_true, probability = y_true[valid], probability[valid]
    if weights is not None:
        weights = np.asarray(weights, dtype=float)[valid]
    return {
        "n": int(len(y_true)),
        "auc": float(roc_auc_score(y_true, probability, sample_weight=weights)),
        "average_precision": float(
            average_precision_score(y_true, probability, sample_weight=weights)
        ),
    }


def fit_predict_two_stage(train_df, test_df, features, fold_number):
    clf = LGBMClassifier(
        objective="binary",
        n_estimators=450,
        learning_rate=0.035,
        num_leaves=31,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_SEED + fold_number,
        n_jobs=4,
        verbose=-1,
    )
    clf.fit(
        train_df[features],
        train_df["presence"],
        categorical_feature=["exclusion_class"],
    )
    raw_probability = clf.predict_proba(test_df[features])[:, 1]
    calibrated_probability = adjust_probability_prior_shift(
        raw_probability,
        float(train_df["presence"].mean()),
        target_prior,
    )

    positive_train = train_df[train_df["presence"].eq(1)]
    reg = LGBMRegressor(
        objective="regression",
        n_estimators=550,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=60,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_SEED + fold_number,
        n_jobs=4,
        verbose=-1,
    )
    reg.fit(
        positive_train[features],
        positive_train["cropland_fraction"],
        categorical_feature=["exclusion_class"],
    )
    conditional = np.clip(reg.predict(test_df[features]), 0, 1)
    expected = calibrated_probability * conditional
    return raw_probability, calibrated_probability, conditional, expected




## 【セル5】raw crop-potential + WXモデルのOOF再推定

crop-potentialはraw値、周辺変数は既存のlog/WX仕様を使います。


In [ ]:
# ============================================================
# 追加セル
# Crop potentialをraw値＋存在フラグで再学習
# raw_crop と raw_crop_wx を既存モデルと比較
# ============================================================

print("=" * 70)
print("RAW CROP-POTENTIAL RE-ESTIMATION")
print("=" * 70)

required_objects = [
    "analysis_sample",
    "splits",
    "fit_predict_two_stage",
    "regression_summary",
    "classification_summary",
    "feature_cols_by_model",
    "wx_feature_cols",
    "base_feature_cols",
    "area_weight",
    "weight_specs",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"以下の変数がありません: {missing}\n"
        "先にセル1〜12を実行してください。"
    )


# ------------------------------------------------------------
# 1. 既存のanalysis_sampleをコピー
# ------------------------------------------------------------

raw_analysis_sample = analysis_sample.copy()

raw_rows = raw_analysis_sample["row"].to_numpy(dtype=int)
raw_cols = raw_analysis_sample["col"].to_numpy(dtype=int)


def sample_raster(array):
    return np.asarray(
        array[raw_rows, raw_cols],
        dtype=np.float32,
    )


def clean_positive_feature(values):
    """
    正の値だけをraw値として残す。
    非正値は0にする。
    同時に、存在フラグと欠測フラグも返す。
    """
    values = np.asarray(values, dtype=np.float32)

    finite = np.isfinite(values)
    exists = finite & (values > 0)
    missing = ~finite

    clean = np.where(
        exists,
        values,
        0.0,
    ).astype(np.float32)

    return (
        clean,
        exists.astype(np.uint8),
        missing.astype(np.uint8),
    )


# ------------------------------------------------------------
# 2. crop potentialのraw値を作成
# ------------------------------------------------------------

rainfed_value_sample = sample_raster(rainfed_value_top5)
irrigated_value_sample = sample_raster(irrigated_value_top5)

rainfed_calorie_sample = sample_raster(rainfed_calorie_top5)
irrigated_calorie_sample = sample_raster(irrigated_calorie_top5)


(
    rainfed_value_raw,
    rainfed_value_exists,
    rainfed_value_missing,
) = clean_positive_feature(rainfed_value_sample)

(
    rainfed_calorie_raw,
    rainfed_calorie_exists,
    rainfed_calorie_missing,
) = clean_positive_feature(rainfed_calorie_sample)


# ------------------------------------------------------------
# 3. 灌漑gainと符号付き差分を作成
# ------------------------------------------------------------

value_pair_valid = (
    np.isfinite(rainfed_value_sample)
    & np.isfinite(irrigated_value_sample)
)

calorie_pair_valid = (
    np.isfinite(rainfed_calorie_sample)
    & np.isfinite(irrigated_calorie_sample)
)


# 符号付き差分
irrigation_value_delta_signed = np.where(
    value_pair_valid,
    irrigated_value_sample - rainfed_value_sample,
    0.0,
).astype(np.float32)

irrigation_calorie_delta_signed = np.where(
    calorie_pair_valid,
    irrigated_calorie_sample - rainfed_calorie_sample,
    0.0,
).astype(np.float32)


# 非負gain
irrigation_value_gain_raw = np.maximum(
    irrigation_value_delta_signed,
    0.0,
).astype(np.float32)

irrigation_calorie_gain_raw = np.maximum(
    irrigation_calorie_delta_signed,
    0.0,
).astype(np.float32)


# gainの存在フラグ
irrigation_value_gain_exists = (
    value_pair_valid
    & (irrigation_value_delta_signed > 0)
).astype(np.uint8)

irrigation_calorie_gain_exists = (
    calorie_pair_valid
    & (irrigation_calorie_delta_signed > 0)
).astype(np.uint8)


# 欠測フラグ
irrigation_value_gain_missing = (
    ~value_pair_valid
).astype(np.uint8)

irrigation_calorie_gain_missing = (
    ~calorie_pair_valid
).astype(np.uint8)


# ------------------------------------------------------------
# 4. DataFrameに追加
# ------------------------------------------------------------

raw_analysis_sample[
    "rainfed_value_top5_raw"
] = rainfed_value_raw

raw_analysis_sample[
    "rainfed_calorie_top5_raw"
] = rainfed_calorie_raw

raw_analysis_sample[
    "irrigation_value_gain_top5_raw"
] = irrigation_value_gain_raw

raw_analysis_sample[
    "irrigation_calorie_gain_top5_raw"
] = irrigation_calorie_gain_raw


raw_analysis_sample[
    "rainfed_value_top5_exists"
] = rainfed_value_exists

raw_analysis_sample[
    "rainfed_calorie_top5_exists"
] = rainfed_calorie_exists

raw_analysis_sample[
    "irrigation_value_gain_top5_exists"
] = irrigation_value_gain_exists

raw_analysis_sample[
    "irrigation_calorie_gain_top5_exists"
] = irrigation_calorie_gain_exists


raw_analysis_sample[
    "irrigation_value_delta_top5_signed"
] = irrigation_value_delta_signed

raw_analysis_sample[
    "irrigation_calorie_delta_top5_signed"
] = irrigation_calorie_delta_signed


raw_analysis_sample[
    "rainfed_value_top5_missing"
] = rainfed_value_missing

raw_analysis_sample[
    "rainfed_calorie_top5_missing"
] = rainfed_calorie_missing

raw_analysis_sample[
    "irrigation_value_gain_top5_missing"
] = irrigation_value_gain_missing

raw_analysis_sample[
    "irrigation_calorie_gain_top5_missing"
] = irrigation_calorie_gain_missing


# ------------------------------------------------------------
# 5. 新しいcrop potential feature list
# ------------------------------------------------------------

non_crop_base_cols = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_min",
]

# 実際の列名を確認
if "log_distance_river_gt10_min" not in raw_analysis_sample.columns:
    non_crop_base_cols[-1] = "log_distance_river_gt10_2020"


raw_crop_cols = [
    # rawの大きさ
    "rainfed_value_top5_raw",
    "rainfed_calorie_top5_raw",
    "irrigation_value_gain_top5_raw",
    "irrigation_calorie_gain_top5_raw",

    # 0か正値か
    "rainfed_value_top5_exists",
    "rainfed_calorie_top5_exists",
    "irrigation_value_gain_top5_exists",
    "irrigation_calorie_gain_top5_exists",

    # 灌漑による符号付き差分
    "irrigation_value_delta_top5_signed",
    "irrigation_calorie_delta_top5_signed",

    # 欠測フラグ
    "rainfed_value_top5_missing",
    "rainfed_calorie_top5_missing",
    "irrigation_value_gain_top5_missing",
    "irrigation_calorie_gain_top5_missing",
]


raw_crop_feature_cols = (
    non_crop_base_cols
    + raw_crop_cols
)


# ------------------------------------------------------------
# 6. raw版のcrop potential WXを作る
# ------------------------------------------------------------

# 既存のraw crop-potential WXキャッシュを読み込むだけ
# キャッシュがない場合は、ここでは再計算せずエラーにする。

raw_crop_wx_feature_cols = []

for radius_km in WX_RADII_KM:
    raw_wx_names = [
        f"wx_{radius_km}km_rainfed_value_top5_raw",
        f"wx_{radius_km}km_rainfed_value_exists_share",
    ]

    for wx_name in raw_wx_names:
        cache_path = OUTPUT_DIR / f"{wx_name}.npy"

        if not cache_path.exists():
            raise FileNotFoundError(
                "raw WXキャッシュがありません。再計算は行わず停止します: "
                f"{cache_path}"
            )

        print("load raw WX cache:", cache_path.name)
        wx_raster = np.load(cache_path, mmap_mode="r")

        if wx_raster.shape != SHAPE:
            raise ValueError(
                f"raw WXキャッシュのshapeが不正です: {cache_path} "
                f"{wx_raster.shape} != {SHAPE}"
            )

        raw_analysis_sample[wx_name] = np.asarray(
            wx_raster[raw_rows, raw_cols],
            dtype=np.float32,
        )
        raw_crop_wx_feature_cols.append(wx_name)


# 既存WXから、log版rainfed value WXだけ除外
non_crop_wx_feature_cols = [
    col
    for col in wx_feature_cols
    if "log_rainfed_value_top5" not in col
]


raw_crop_wx_feature_cols = (
    raw_crop_feature_cols
    + non_crop_wx_feature_cols
    + raw_crop_wx_feature_cols
)


feature_cols_raw_by_model = {
    "raw_crop": raw_crop_feature_cols,
    "raw_crop_wx": raw_crop_wx_feature_cols,
}


print()
print("raw_crop feature count:", len(raw_crop_feature_cols))
print("raw_crop_wx feature count:", len(raw_crop_wx_feature_cols))
print()
print("raw_crop features:")
print(raw_crop_feature_cols)
print()
print("raw_crop_wx features:")
print(raw_crop_wx_feature_cols)


# ------------------------------------------------------------
# 7. raw版Spatial OOF
# ------------------------------------------------------------

raw_oof = {
    model_name: {
        "fold": np.full(
            len(raw_analysis_sample),
            -1,
            dtype=np.int16,
        ),
        "presence_raw": np.full(
            len(raw_analysis_sample),
            np.nan,
            dtype=np.float32,
        ),
        "presence_calibrated": np.full(
            len(raw_analysis_sample),
            np.nan,
            dtype=np.float32,
        ),
        "fraction_conditional": np.full(
            len(raw_analysis_sample),
            np.nan,
            dtype=np.float32,
        ),
        "fraction_expected": np.full(
            len(raw_analysis_sample),
            np.nan,
            dtype=np.float32,
        ),
    }
    for model_name in feature_cols_raw_by_model
}


raw_fold_rows = []

for fold_number, (train_idx, test_idx) in enumerate(
    splits,
    start=1,
):

    train_df = raw_analysis_sample.iloc[train_idx]
    test_df = raw_analysis_sample.iloc[test_idx]

    print()
    print("=" * 70)
    print(
        f"RAW MODEL FOLD {fold_number}/{N_SPLITS}"
    )
    print("=" * 70)

    for model_name, features in feature_cols_raw_by_model.items():

        print(
            f"training: {model_name} "
            f"({len(features)} features)"
        )

        raw_probability, calibrated_probability, conditional, expected = (
            fit_predict_two_stage(
                train_df,
                test_df,
                features,
                fold_number,
            )
        )

        raw_oof[model_name]["fold"][test_idx] = (
            fold_number
        )

        raw_oof[model_name]["presence_raw"][test_idx] = (
            raw_probability
        )

        raw_oof[model_name]["presence_calibrated"][test_idx] = (
            calibrated_probability
        )

        raw_oof[model_name]["fraction_conditional"][test_idx] = (
            conditional
        )

        raw_oof[model_name]["fraction_expected"][test_idx] = (
            expected
        )

        y_test = test_df[
            "presence"
        ].to_numpy(dtype=np.uint8)

        y_fraction = test_df[
            "cropland_fraction"
        ].to_numpy(dtype=float)

        positive_mask = y_test == 1

        combined_metrics = regression_summary(
            y_fraction,
            expected,
        )

        conditional_metrics = regression_summary(
            y_fraction[positive_mask],
            conditional[positive_mask],
        )

        stage1_metrics = classification_summary(
            y_test,
            raw_probability,
        )

        raw_fold_rows.append({
            "model": model_name,
            "fold": fold_number,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "presence_auc": stage1_metrics["auc"],
            "presence_average_precision": (
                stage1_metrics["average_precision"]
            ),
            "combined_r2": combined_metrics["r2"],
            "combined_rmse": combined_metrics["rmse"],
            "combined_mae": combined_metrics["mae"],
            "conditional_r2": conditional_metrics["r2"],
            "conditional_rmse": conditional_metrics["rmse"],
            "conditional_mae": conditional_metrics["mae"],
        })

        del (
            raw_probability,
            calibrated_probability,
            conditional,
            expected,
        )

        gc.collect()


raw_fold_metrics_df = pd.DataFrame(
    raw_fold_rows
)




## 【セル6】レジーム別モデルの空間OOF比較

K=2・3・4の候補を比較します。レジーム分類は各foldのtrainデータだけで作り、testデータに適用します。


In [ ]:
# ============================================================
# 【セル24】農業レジーム分割モデルの空間OOF比較
# ============================================================

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder


print("=" * 80)
print("REGIME-SPECIFIC MODEL VS POOLED MODEL")
print("=" * 80)


# ------------------------------------------------------------
# 0. 前提確認
# ------------------------------------------------------------

_required_objects = [
    "raw_analysis_sample",
    "raw_oof",
    "feature_cols_raw_by_model",
    "splits",
    "fit_predict_two_stage",
    "adjust_probability_prior_shift",
]

_missing_objects = [
    name for name in _required_objects
    if name not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "先にセル21・23を実行してください。"
        f"不足している変数: {_missing_objects}"
    )


REGIME_MODEL_NAME = "raw_crop_wx"

# 最初は2〜4で十分です。
# 必要なら後で5を追加してください。
REGIME_K_VALUES = (2, 3, 4)

REGIME_RANDOM_SEED = RANDOM_SEED + 5000
REGIME_MIN_TRAIN_ROWS = 2_000
REGIME_MIN_TRAIN_POSITIVE = 200
REGIME_MIN_TRAIN_NEGATIVE = 200

REGIME_OUTPUT_DIR = OUTPUT_DIR / "regime_model_comparison"
REGIME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. レジーム分類に使う変数
# ------------------------------------------------------------

# raw_crop_wxの説明変数を候補にする。
# ただし、データの有無を表す変数はレジーム分類から除外する。
# これは「データがある地域」と「農業レジーム」を混同しないため。
all_model_features = list(
    feature_cols_raw_by_model[REGIME_MODEL_NAME]
)

REGIME_CATEGORICAL_COLS = [
    col
    for col in ["exclusion_class"]
    if col in all_model_features
]

REGIME_NUMERIC_COLS = [
    col
    for col in all_model_features
    if col not in REGIME_CATEGORICAL_COLS
    and "exists" not in col.lower()
    and "missing" not in col.lower()
]

REGIME_FEATURE_COLS = (
    REGIME_NUMERIC_COLS
    + REGIME_CATEGORICAL_COLS
)

if len(REGIME_NUMERIC_COLS) == 0:
    raise RuntimeError("レジーム分類用の数値変数がありません。")

print()
print("Regime numeric features:")
print(REGIME_NUMERIC_COLS)

print()
print("Regime categorical features:")
print(REGIME_CATEGORICAL_COLS)


# ------------------------------------------------------------
# 2. レジーム分類用の前処理
# ------------------------------------------------------------

def make_regime_matrices(train_df, test_df):
    """
    trainデータだけで標準化・カテゴリ変換を学習し、
    testデータに適用する。
    """

    X_train_num = train_df[
        REGIME_NUMERIC_COLS
    ].to_numpy(dtype=np.float32)

    X_test_num = test_df[
        REGIME_NUMERIC_COLS
    ].to_numpy(dtype=np.float32)

    if not np.isfinite(X_train_num).all():
        raise ValueError(
            "レジーム分類用train変数にNaNまたはinfがあります。"
        )

    if not np.isfinite(X_test_num).all():
        raise ValueError(
            "レジーム分類用test変数にNaNまたはinfがあります。"
        )

    scaler = StandardScaler()

    X_train_num = scaler.fit_transform(
        X_train_num
    ).astype(np.float32)

    X_test_num = scaler.transform(
        X_test_num
    ).astype(np.float32)

    matrices_train = [X_train_num]
    matrices_test = [X_test_num]

    if len(REGIME_CATEGORICAL_COLS) > 0:

        train_cat = train_df[
            REGIME_CATEGORICAL_COLS
        ].astype(str)

        test_cat = test_df[
            REGIME_CATEGORICAL_COLS
        ].astype(str)

        try:
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            )
        except TypeError:
            # 古いscikit-learn用
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse=False,
            )

        X_train_cat = encoder.fit_transform(
            train_cat
        ).astype(np.float32)

        X_test_cat = encoder.transform(
            test_cat
        ).astype(np.float32)

        matrices_train.append(X_train_cat)
        matrices_test.append(X_test_cat)

    X_train = np.hstack(
        matrices_train
    ).astype(np.float32)

    X_test = np.hstack(
        matrices_test
    ).astype(np.float32)

    return X_train, X_test


# ------------------------------------------------------------
# 3. レジーム内の存在確率を補正するための事前確率
# ------------------------------------------------------------

def estimate_regime_target_prior(
    train_df,
    train_global_indices,
):
    """
    レジーム内の農地存在確率を推定する。

    area_weightがある場合は面積加重、
    ない場合はtrain内の単純平均を使用する。
    """

    y = train_df[
        "presence"
    ].to_numpy(dtype=float)

    if area_weight is None:
        return float(np.mean(y))

    w = np.asarray(
        area_weight,
        dtype=float,
    )[train_global_indices]

    valid = (
        np.isfinite(y)
        & np.isfinite(w)
        & (w > 0)
    )

    if valid.sum() == 0:
        return float(np.mean(y))

    return float(
        np.average(
            y[valid],
            weights=w[valid],
        )
    )


# ------------------------------------------------------------
# 4. レジーム内用の二段階LightGBM
# ------------------------------------------------------------

def fit_predict_two_stage_regime(
    train_df,
    test_df,
    features,
    fold_number,
    regime_id,
    target_prior_regime,
):

    categorical_features = [
        col
        for col in ["exclusion_class"]
        if col in features
    ]

    random_state = (
        RANDOM_SEED
        + fold_number * 1000
        + regime_id
    )

    # -------------------------
    # Stage 1: presence
    # -------------------------

    clf = LGBMClassifier(
        objective="binary",
        n_estimators=450,
        learning_rate=0.035,
        num_leaves=31,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=random_state,
        n_jobs=4,
        verbose=-1,
    )

    clf.fit(
        train_df[features],
        train_df["presence"],
        categorical_feature=categorical_features,
    )

    raw_probability = clf.predict_proba(
        test_df[features]
    )[:, 1]

    train_sample_prior = float(
        train_df["presence"].mean()
    )

    calibrated_probability = (
        adjust_probability_prior_shift(
            raw_probability,
            train_sample_prior,
            target_prior_regime,
        )
    )

    # -------------------------
    # Stage 2: conditional fraction
    # -------------------------

    positive_train = train_df[
        train_df["presence"].eq(1)
    ]

    reg = LGBMRegressor(
        objective="regression",
        n_estimators=550,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=60,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=random_state + 100,
        n_jobs=4,
        verbose=-1,
    )

    reg.fit(
        positive_train[features],
        positive_train["cropland_fraction"],
        categorical_feature=categorical_features,
    )

    conditional_fraction = np.clip(
        reg.predict(test_df[features]),
        0,
        1,
    )

    expected_fraction = (
        calibrated_probability
        * conditional_fraction
    )

    return (
        raw_probability,
        calibrated_probability,
        conditional_fraction,
        expected_fraction,
    )


# ------------------------------------------------------------
# 5. 評価関数
# ------------------------------------------------------------

def regime_regression_metrics(
    y_true,
    y_pred,
    weights=None,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    if weights is None:
        weights = np.ones(
            len(y_true),
            dtype=float,
        )
    else:
        weights = np.asarray(
            weights,
            dtype=float,
        )

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & np.isfinite(weights)
        & (weights > 0)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]
    weights = weights[valid]

    mean_true = np.average(
        y_true,
        weights=weights,
    )

    residual = y_true - y_pred

    ss_res = np.sum(
        weights * residual**2
    )

    ss_tot = np.sum(
        weights * (y_true - mean_true)**2
    )

    return {
        "n": int(len(y_true)),
        "r2": float(
            1 - ss_res / ss_tot
        ) if ss_tot > 0 else np.nan,
        "rmse": float(
            np.sqrt(
                np.average(
                    residual**2,
                    weights=weights,
                )
            )
        ),
        "mae": float(
            np.average(
                np.abs(residual),
                weights=weights,
            )
        ),
        "observed_mean": float(mean_true),
        "predicted_mean": float(
            np.average(
                y_pred,
                weights=weights,
            )
        ),
    }


# ------------------------------------------------------------
# 6. データ・OOF予測の準備
# ------------------------------------------------------------

y_fraction_regime = raw_analysis_sample[
    "cropland_fraction"
].to_numpy(dtype=float)

pooled_prediction = raw_oof[
    REGIME_MODEL_NAME
]["fraction_expected"]

pooled_fold = raw_oof[
    REGIME_MODEL_NAME
]["fold"]

if len(raw_analysis_sample) != len(pooled_prediction):
    raise RuntimeError(
        "raw_analysis_sampleとraw_oofの行数が一致していません。"
    )

weighting_specs_regime = {
    "unweighted_sample": None,
}

if area_weight is not None:
    weighting_specs_regime[
        "area_weighted_reweighted"
    ] = np.asarray(
        area_weight,
        dtype=float,
    )


# ------------------------------------------------------------
# 7. Kごとにレジーム別モデルをOOF推定
# ------------------------------------------------------------

regime_predictions = {}
regime_labels = {}

regime_distribution_rows = []

for n_regimes in REGIME_K_VALUES:

    print()
    print("=" * 80)
    print(
        f"REGIME-SPECIFIC OOF MODEL: "
        f"K={n_regimes}"
    )
    print("=" * 80)

    regime_prediction = np.full(
        len(raw_analysis_sample),
        np.nan,
        dtype=np.float32,
    )

    regime_label_oof = np.full(
        len(raw_analysis_sample),
        -1,
        dtype=np.int16,
    )

    candidate_distribution_rows = []
    candidate_failed = False
    failure_reason = None

    for fold_number, (
        train_idx,
        test_idx,
    ) in enumerate(
        splits,
        start=1,
    ):

        print()
        print(
            f"Fold {fold_number}/{N_SPLITS}"
        )

        train_df = raw_analysis_sample.iloc[
            train_idx
        ]

        test_df = raw_analysis_sample.iloc[
            test_idx
        ]

        # train foldだけでレジーム分類用変換を学習
        X_train_regime, X_test_regime = (
            make_regime_matrices(
                train_df,
                test_df,
            )
        )

        cluster_model = MiniBatchKMeans(
            n_clusters=n_regimes,
            random_state=(
                REGIME_RANDOM_SEED
                + fold_number
                + n_regimes * 100
            ),
            batch_size=8192,
            n_init=10,
            max_iter=100,
            reassignment_ratio=0.01,
        )

        train_regime_labels = (
            cluster_model.fit_predict(
                X_train_regime
            )
        )

        test_regime_labels = (
            cluster_model.predict(
                X_test_regime
            )
        )

        # 各レジームのtrainデータ量を確認
        for regime_id in range(n_regimes):

            train_regime_mask = (
                train_regime_labels
                == regime_id
            )

            test_regime_mask = (
                test_regime_labels
                == regime_id
            )

            train_n = int(
                train_regime_mask.sum()
            )

            test_n = int(
                test_regime_mask.sum()
            )

            train_positive_n = int(
                (
                    train_regime_mask
                    & (
                        train_df[
                            "presence"
                        ].to_numpy()
                        == 1
                    )
                ).sum()
            )

            train_negative_n = int(
                (
                    train_regime_mask
                    & (
                        train_df[
                            "presence"
                        ].to_numpy()
                        == 0
                    )
                ).sum()
            )

            candidate_distribution_rows.append({
                "k": n_regimes,
                "fold": fold_number,
                "regime_id": regime_id,
                "train_n": train_n,
                "test_n": test_n,
                "train_positive_n": train_positive_n,
                "train_negative_n": train_negative_n,
            })

            if test_n == 0:
                continue

            if train_n < REGIME_MIN_TRAIN_ROWS:
                candidate_failed = True
                failure_reason = (
                    f"K={n_regimes}, fold={fold_number}, "
                    f"regime={regime_id}: "
                    f"train_n={train_n}"
                )

            if train_positive_n < (
                REGIME_MIN_TRAIN_POSITIVE
            ):
                candidate_failed = True
                failure_reason = (
                    f"K={n_regimes}, fold={fold_number}, "
                    f"regime={regime_id}: "
                    f"positive_n={train_positive_n}"
                )

            if train_negative_n < (
                REGIME_MIN_TRAIN_NEGATIVE
            ):
                candidate_failed = True
                failure_reason = (
                    f"K={n_regimes}, fold={fold_number}, "
                    f"regime={regime_id}: "
                    f"negative_n={train_negative_n}"
                )

        if candidate_failed:
            break

        # レジームごとに別々の二段階モデルを学習
        for regime_id in range(n_regimes):

            train_regime_mask = (
                train_regime_labels
                == regime_id
            )

            test_regime_mask = (
                test_regime_labels
                == regime_id
            )

            if not test_regime_mask.any():
                continue

            regime_train_df = train_df.iloc[
                train_regime_mask
            ]

            regime_test_df = test_df.iloc[
                test_regime_mask
            ]

            regime_train_global_indices = (
                train_idx[train_regime_mask]
            )

            regime_target_prior = (
                estimate_regime_target_prior(
                    regime_train_df,
                    regime_train_global_indices,
                )
            )

            (
                raw_probability,
                calibrated_probability,
                conditional_fraction,
                expected_fraction,
            ) = fit_predict_two_stage_regime(
                regime_train_df,
                regime_test_df,
                feature_cols_raw_by_model[
                    REGIME_MODEL_NAME
                ],
                fold_number,
                regime_id,
                regime_target_prior,
            )

            target_global_indices = (
                test_idx[test_regime_mask]
            )

            regime_prediction[
                target_global_indices
            ] = expected_fraction

            regime_label_oof[
                target_global_indices
            ] = regime_id

            del (
                raw_probability,
                calibrated_probability,
                conditional_fraction,
                expected_fraction,
            )

        del (
            X_train_regime,
            X_test_regime,
            cluster_model,
        )

        gc.collect()

    if candidate_failed:
        print()
        print(
            f"SKIP K={n_regimes}: "
            f"{failure_reason}"
        )
        continue

    if not np.isfinite(
        regime_prediction
    ).all():
        raise RuntimeError(
            f"K={n_regimes}でOOF予測が完成していません。"
        )

    if (regime_label_oof < 0).any():
        raise RuntimeError(
            f"K={n_regimes}でレジームラベルが未割当です。"
        )

    regime_predictions[
        n_regimes
    ] = regime_prediction

    regime_labels[
        n_regimes
    ] = regime_label_oof

    regime_distribution_rows.extend(
        candidate_distribution_rows
    )

    print(
        f"K={n_regimes} completed."
    )


# ------------------------------------------------------------
# 8. pooledモデルとレジーム別モデルの比較
# ------------------------------------------------------------

candidate_metrics_rows = []

for n_regimes, regime_prediction in (
    regime_predictions.items()
):

    for weighting_name, weights in (
        weighting_specs_regime.items()
    ):

        regime_result = regime_regression_metrics(
            y_fraction_regime,
            regime_prediction,
            weights=weights,
        )

        pooled_result = regime_regression_metrics(
            y_fraction_regime,
            pooled_prediction,
            weights=weights,
        )

        candidate_metrics_rows.append({
            "k": n_regimes,
            "weighting": weighting_name,
            "regime_model_r2": regime_result["r2"],
            "pooled_model_r2": pooled_result["r2"],
            "delta_r2": (
                regime_result["r2"]
                - pooled_result["r2"]
            ),
            "regime_model_rmse": regime_result["rmse"],
            "pooled_model_rmse": pooled_result["rmse"],
            "delta_rmse": (
                regime_result["rmse"]
                - pooled_result["rmse"]
            ),
            "regime_model_mae": regime_result["mae"],
            "pooled_model_mae": pooled_result["mae"],
            "delta_mae": (
                regime_result["mae"]
                - pooled_result["mae"]
            ),
        })


regime_candidate_metrics_df = pd.DataFrame(
    candidate_metrics_rows
)

print()
print("=" * 80)
print("POOLED VS REGIME-SPECIFIC GLOBAL METRICS")
print("=" * 80)

print(
    regime_candidate_metrics_df[
        [
            "k",
            "weighting",
            "regime_model_r2",
            "pooled_model_r2",
            "delta_r2",
            "regime_model_rmse",
            "pooled_model_rmse",
            "delta_rmse",
            "regime_model_mae",
            "pooled_model_mae",
            "delta_mae",
        ]
    ].round(6).to_string(index=False)
)


# ------------------------------------------------------------
# 9. fold別の比較
# ------------------------------------------------------------

regime_fold_metrics_rows = []

for n_regimes, regime_prediction in (
    regime_predictions.items()
):

    for fold_number in range(
        1,
        N_SPLITS + 1,
    ):

        fold_mask = (
            pooled_fold
            == fold_number
        )

        for weighting_name, weights in (
            weighting_specs_regime.items()
        ):

            if weights is None:
                fold_weights = None
            else:
                fold_weights = weights[
                    fold_mask
                ]

            regime_result = regime_regression_metrics(
                y_fraction_regime[
                    fold_mask
                ],
                regime_prediction[
                    fold_mask
                ],
                weights=fold_weights,
            )

            pooled_result = regime_regression_metrics(
                y_fraction_regime[
                    fold_mask
                ],
                pooled_prediction[
                    fold_mask
                ],
                weights=fold_weights,
            )

            regime_fold_metrics_rows.append({
                "k": n_regimes,
                "fold": fold_number,
                "weighting": weighting_name,
                "regime_model_r2": regime_result["r2"],
                "pooled_model_r2": pooled_result["r2"],
                "delta_r2": (
                    regime_result["r2"]
                    - pooled_result["r2"]
                ),
                "regime_model_rmse": regime_result["rmse"],
                "pooled_model_rmse": pooled_result["rmse"],
                "delta_rmse": (
                    regime_result["rmse"]
                    - pooled_result["rmse"]
                ),
                "regime_model_mae": regime_result["mae"],
                "pooled_model_mae": pooled_result["mae"],
                "delta_mae": (
                    regime_result["mae"]
                    - pooled_result["mae"]
                ),
            })


regime_fold_metrics_df = pd.DataFrame(
    regime_fold_metrics_rows
)

print()
print("=" * 80)
print("FOLD-BY-FOLD DELTA R2")
print("=" * 80)

print(
    regime_fold_metrics_df[
        [
            "k",
            "fold",
            "weighting",
            "regime_model_r2",
            "pooled_model_r2",
            "delta_r2",
        ]
    ].round(6).to_string(index=False)
)


# ------------------------------------------------------------
# 10. fold平均・標準偏差
# ------------------------------------------------------------

regime_fold_summary_df = (
    regime_fold_metrics_df
    .groupby(
        ["k", "weighting"],
        as_index=False,
    )
    .agg(
        mean_regime_r2=(
            "regime_model_r2",
            "mean",
        ),
        sd_regime_r2=(
            "regime_model_r2",
            "std",
        ),
        mean_pooled_r2=(
            "pooled_model_r2",
            "mean",
        ),
        mean_delta_r2=(
            "delta_r2",
            "mean",
        ),
        sd_delta_r2=(
            "delta_r2",
            "std",
        ),
        min_delta_r2=(
            "delta_r2",
            "min",
        ),
        max_delta_r2=(
            "delta_r2",
            "max",
        ),
    )
)

print()
print("=" * 80)
print("FOLD-MEAN SUMMARY")
print("=" * 80)

print(
    regime_fold_summary_df.round(6).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 11. レジーム別のOOF結果を保存
# ------------------------------------------------------------

regime_oof_table = raw_analysis_sample[
    [
        "row",
        "col",
        "lat",
        "lon",
        "spatial_block",
        "presence",
        "cropland_fraction",
    ]
].copy()

regime_oof_table[
    "pooled_raw_crop_wx_prediction"
] = pooled_prediction

regime_oof_table[
    "pooled_raw_crop_wx_residual"
] = (
    regime_oof_table[
        "cropland_fraction"
    ].to_numpy(dtype=float)
    - pooled_prediction
)

for n_regimes, prediction in (
    regime_predictions.items()
):

    regime_oof_table[
        f"regime_k{n_regimes}_prediction"
    ] = prediction

    regime_oof_table[
        f"regime_k{n_regimes}_residual"
    ] = (
        regime_oof_table[
            "cropland_fraction"
        ].to_numpy(dtype=float)
        - prediction
    )

    regime_oof_table[
        f"regime_k{n_regimes}_id"
    ] = regime_labels[
        n_regimes
    ]


regime_distribution_df = pd.DataFrame(
    regime_distribution_rows
)


# ------------------------------------------------------------
# 12. 保存
# ------------------------------------------------------------

regime_candidate_metrics_df.to_csv(
    REGIME_OUTPUT_DIR
    / "regime_candidate_global_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

regime_fold_metrics_df.to_csv(
    REGIME_OUTPUT_DIR
    / "regime_candidate_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

regime_fold_summary_df.to_csv(
    REGIME_OUTPUT_DIR
    / "regime_candidate_fold_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

regime_oof_table.to_csv(
    REGIME_OUTPUT_DIR
    / "regime_candidate_oof_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

regime_distribution_df.to_csv(
    REGIME_OUTPUT_DIR
    / "regime_candidate_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 13. 最も良いKを表示
# ------------------------------------------------------------

print()
print("=" * 80)
print("BEST REGIME COUNT BY WEIGHTING")
print("=" * 80)

for weighting_name in weighting_specs_regime:

    subset = regime_candidate_metrics_df[
        regime_candidate_metrics_df[
            "weighting"
        ] == weighting_name
    ].copy()

    if len(subset) == 0:
        continue

    best_row = subset.loc[
        subset["delta_r2"].idxmax()
    ]

    print()
    print(
        f"[{weighting_name}]"
    )
    print(
        f"best K: {int(best_row['k'])}"
    )
    print(
        f"pooled R2: "
        f"{best_row['pooled_model_r2']:.6f}"
    )
    print(
        f"regime-specific R2: "
        f"{best_row['regime_model_r2']:.6f}"
    )
    print(
        f"delta R2: "
        f"{best_row['delta_r2']:.6f}"
    )

print()
print("保存先:")
print(REGIME_OUTPUT_DIR)

## 【セル7】最良候補レジームの地理的可視化

OOF精度が最も高かった候補Kを自動選択し、そのKについて全サンプルでレジームを再推定して地理分布を表示します。

このセルのクラスタラベルは可視化用です。レジーム別OOF精度の判定は、前セルのOOF結果を使います。


In [ ]:
# ============================================================
# 【セル7】最良候補レジームの地理的可視化
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.cluster import MiniBatchKMeans


if "regime_candidate_metrics_df" not in globals():
    raise RuntimeError(
        "先に【セル6】レジーム別モデルの空間OOF比較を実行してください。"
    )


# ------------------------------------------------------------
# 1. 最も成績が良かった候補Kを選択
# ------------------------------------------------------------

if area_weight is not None:
    PLOT_SELECTION_WEIGHTING = "area_weighted_reweighted"
else:
    PLOT_SELECTION_WEIGHTING = "unweighted_sample"

selection_table = regime_candidate_metrics_df[
    regime_candidate_metrics_df["weighting"]
    == PLOT_SELECTION_WEIGHTING
].copy()

if len(selection_table) == 0:
    raise RuntimeError(
        "レジーム候補の評価結果が見つかりません。"
    )

best_row = selection_table.loc[
    selection_table["delta_r2"].idxmax()
]

BEST_REGIME_K_FOR_PLOT = int(best_row["k"])

print("=" * 80)
print("BEST CANDIDATE REGIME FOR VISUALIZATION")
print("=" * 80)
print(
    f"weighting: {PLOT_SELECTION_WEIGHTING}"
)
print(
    f"best candidate K: {BEST_REGIME_K_FOR_PLOT}"
)
print(
    f"pooled R2: {best_row['pooled_model_r2']:.6f}"
)
print(
    f"regime-specific R2: "
    f"{best_row['regime_model_r2']:.6f}"
)
print(
    f"delta R2: {best_row['delta_r2']:.6f}"
)
print()
print(
    "注意：ここで選ばれるのは、試した候補の中で"
    "最も悪化幅が小さいKです。"
)


# ------------------------------------------------------------
# 2. 全サンプルで可視化用の安定したレジームラベルを作る
# ------------------------------------------------------------

# OOF中はfoldごとにクラスタ番号が入れ替わり得るため、
# 可視化用には全サンプルで同じクラスタリングを一度だけ行う。
# これは精度評価ではなく、地理的解釈のための再推定である。

plot_base_df = raw_analysis_sample.copy()

X_plot, _ = make_regime_matrices(
    plot_base_df,
    plot_base_df,
)

plot_cluster_model = MiniBatchKMeans(
    n_clusters=BEST_REGIME_K_FOR_PLOT,
    random_state=REGIME_RANDOM_SEED + 9000,
    batch_size=8192,
    n_init=10,
    max_iter=100,
    reassignment_ratio=0.01,
)

plot_labels_raw = plot_cluster_model.fit_predict(
    X_plot
)

plot_base_df["regime_raw"] = plot_labels_raw


# クラスタ番号は任意なので、rainfed potentialの平均が高い順に並べ替える。
if "rainfed_value_top5_raw" in plot_base_df.columns:
    regime_sort_col = "rainfed_value_top5_raw"
else:
    regime_sort_col = REGIME_NUMERIC_COLS[0]

regime_sort_order = (
    plot_base_df
    .groupby("regime_raw")[regime_sort_col]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

regime_label_map = {
    old_label: new_label
    for new_label, old_label
    in enumerate(regime_sort_order)
}

plot_base_df["regime"] = plot_base_df[
    "regime_raw"
].map(regime_label_map).astype(int)


# ------------------------------------------------------------
# 3. レジーム別の基本統計
# ------------------------------------------------------------

regime_basic_summary = (
    plot_base_df
    .groupby("regime")
    .agg(
        n=("regime", "size"),
        share=("regime", lambda x: len(x) / len(plot_base_df)),
        mean_cropland_fraction=(
            "cropland_fraction",
            "mean",
        ),
        presence_share=("presence", "mean"),
        mean_lat=("lat", "mean"),
        mean_lon=("lon", "mean"),
    )
    .reset_index()
)

print("レジーム別基本統計")
print(
    regime_basic_summary.round(5).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 4. 地理分布：全レジームを1枚に表示
# ------------------------------------------------------------

plot_rng = np.random.default_rng(
    RANDOM_SEED + 9001
)

max_plot_rows = 120_000

if len(plot_base_df) > max_plot_rows:
    plot_indices = plot_rng.choice(
        len(plot_base_df),
        size=max_plot_rows,
        replace=False,
    )
    plot_points = plot_base_df.iloc[
        plot_indices
    ].copy()
else:
    plot_points = plot_base_df

colors = [
    "#2166ac",
    "#fdae61",
    "#1a9850",
    "#d73027",
    "#762a83",
]

fig, ax = plt.subplots(
    figsize=(16, 7),
)

for regime_id in range(
    BEST_REGIME_K_FOR_PLOT
):

    subset = plot_points[
        plot_points["regime"] == regime_id
    ]

    ax.scatter(
        subset["lon"],
        subset["lat"],
        s=1.0,
        alpha=0.45,
        color=colors[regime_id],
        label=(
            f"Regime {regime_id} "
            f"(n={len(subset):,})"
        ),
        linewidths=0,
    )

ax.set_title(
    f"Geographical distribution of candidate regimes "
    f"(K={BEST_REGIME_K_FOR_PLOT})"
)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.grid(alpha=0.25)
ax.legend(markerscale=6)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. レジーム別の地理分布：各レジームを個別表示
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    BEST_REGIME_K_FOR_PLOT,
    figsize=(7 * BEST_REGIME_K_FOR_PLOT, 5),
    sharex=True,
    sharey=True,
)

if BEST_REGIME_K_FOR_PLOT == 1:
    axes = [axes]

for regime_id, ax in enumerate(axes):

    ax.scatter(
        plot_points["lon"],
        plot_points["lat"],
        s=0.4,
        color="lightgray",
        alpha=0.20,
        linewidths=0,
    )

    subset = plot_points[
        plot_points["regime"] == regime_id
    ]

    ax.scatter(
        subset["lon"],
        subset["lat"],
        s=1.0,
        color=colors[regime_id],
        alpha=0.55,
        linewidths=0,
    )

    row = regime_basic_summary[
        regime_basic_summary["regime"]
        == regime_id
    ].iloc[0]

    ax.set_title(
        f"Regime {regime_id}\n"
        f"share={row['share']:.2%}, "
        f"cropland={row['mean_cropland_fraction']:.3f}"
    )
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.grid(alpha=0.25)
    ax.set_xlabel("longitude")

axes[0].set_ylabel("latitude")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 6. レジームごとの特徴プロファイル
# ------------------------------------------------------------

river_column = next(
    (
        col
        for col in [
            "log_distance_river_gt10_min",
            "log_distance_river_gt10_2020",
        ]
        if col in plot_base_df.columns
    ),
    None,
)

profile_candidates = [
    "elevation_m",
    "slope",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "rainfed_value_top5_raw",
    "irrigation_value_gain_top5_raw",
    "wx_50km_rainfed_value_top5_raw",
    "wx_100km_rainfed_value_top5_raw",
]

if river_column is not None:
    profile_candidates.insert(
        6,
        river_column,
    )

profile_cols = [
    col
    for col in profile_candidates
    if col in plot_base_df.columns
]

profile_mean = (
    plot_base_df
    .groupby("regime")[profile_cols]
    .mean()
)

global_mean = plot_base_df[
    profile_cols
].mean()

global_sd = plot_base_df[
    profile_cols
].std().replace(0, 1)

profile_z = (
    profile_mean - global_mean
).div(global_sd)

print()
print("レジーム別特徴量の平均")
print(profile_mean.round(4).to_string())

fig, ax = plt.subplots(
    figsize=(14, 5),
)

image = ax.imshow(
    profile_z.to_numpy(),
    aspect="auto",
    cmap="coolwarm",
    vmin=-2,
    vmax=2,
)

ax.set_yticks(
    np.arange(len(profile_z.index))
)
ax.set_yticklabels(
    [f"Regime {idx}" for idx in profile_z.index]
)

ax.set_xticks(
    np.arange(len(profile_z.columns))
)
ax.set_xticklabels(
    profile_z.columns,
    rotation=60,
    ha="right",
)

ax.set_title(
    "Regime feature profile "
    "(standardized mean relative to global sample)"
)

fig.colorbar(
    image,
    ax=ax,
    label="standardized mean",
)

plt.tight_layout()
plt.show()
